# Real Estate Investment Advisor
## Phase 1: Data Loading & Cleaning

**Goal:** Load the raw dataset, understand its structure, identify and fix data quality issues, and save a cleaned version for EDA and modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Load the dataset
df = pd.read_csv("india_housing_prices.csv")

print(f"✅ Data loaded successfully")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Data loaded successfully
Shape: 250,000 rows × 23 columns


In [3]:
# First look at the data
print("=== First 5 Rows ===")
display(df.head())

print("\n=== Data Types ===")
print(df.dtypes)

print("\n=== Basic Statistics ===")
display(df.describe())


=== First 5 Rows ===


,ID,State,City,Locality,Property_Type,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Furnished_Status,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals,Public_Transport_Accessibility,Parking_Space,Security,Amenities,Facing,Owner_Type,Availability_Status
0,1,Tamil Nadu,Chennai,Locality_84,Apartment,1,4740,489.7600,0.1000,1990,Furnished,22,1,35,10,3,High,No,No,"Playground, Gym, Garden, Pool, Clubhouse",West,Owner,Ready_to_Move
1,2,Maharashtra,Pune,Locality_490,Independent House,3,2364,195.5200,0.0800,2008,Unfurnished,21,20,17,8,1,Low,No,Yes,"Playground, Clubhouse, Pool, Gym, Garden",North,Builder,Under_Construction
2,3,Punjab,Ludhiana,Locality_167,Apartment,2,3642,183.7900,0.0500,1997,Semi-furnished,19,27,28,9,8,Low,Yes,No,"Clubhouse, Pool, Playground, Gym",South,Broker,Ready_to_Move
3,4,Rajasthan,Jodhpur,Locality_393,Independent House,2,2741,300.2900,0.1100,1991,Furnished,21,26,34,5,7,High,Yes,Yes,"Playground, Clubhouse, Gym, Pool, Garden",North,Builder,Ready_to_Move
4,5,Rajasthan,Jaipur,Locality_466,Villa,4,4823,182.9000,0.0400,2002,Semi-furnished,3,2,23,4,9,Low,No,Yes,"Playground, Garden, Gym, Pool, Clubhouse",East,Builder,Ready_to_Move



=== Data Types ===
ID                                  int64
State                              object
City                               object
Locality                           object
Property_Type                      object
BHK                                 int64
Size_in_SqFt                        int64
Price_in_Lakhs                    float64
Price_per_SqFt                    float64
Year_Built                          int64
Furnished_Status                   object
Floor_No                            int64
Total_Floors                        int64
Age_of_Property                     int64
Nearby_Schools                      int64
Nearby_Hospitals                    int64
Public_Transport_Accessibility     object
Parking_Space                      object
Security                           object
Amenities                          object
Facing                             object
Owner_Type                         object
Availability_Status                object
dtype: object


,ID,BHK,Size_in_SqFt,Price_in_Lakhs,Price_per_SqFt,Year_Built,Floor_No,Total_Floors,Age_of_Property,Nearby_Schools,Nearby_Hospitals
count,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000,250000.0000
mean,125000.5000,2.9994,2749.8132,254.5869,0.1306,2006.5200,14.9668,15.5030,18.4800,5.4999,5.4980
std,72168.9280,1.4155,1300.6070,141.3499,0.1307,9.8086,8.9480,8.6716,9.8086,2.8786,2.8719
min,1.0000,1.0000,500.0000,10.0000,0.0000,1990.0000,0.0000,1.0000,2.0000,1.0000,1.0000
25%,62500.7500,2.0000,1623.0000,132.5500,0.0500,1998.0000,7.0000,8.0000,10.0000,3.0000,3.0000
50%,125000.5000,3.0000,2747.0000,253.8700,0.0900,2007.0000,15.0000,15.0000,18.0000,5.0000,5.0000
75%,187500.2500,4.0000,3874.0000,376.8800,0.1600,2015.0000,23.0000,23.0000,27.0000,8.0000,8.0000
max,250000.0000,5.0000,5000.0000,500.0000,0.9900,2023.0000,30.0000,30.0000,35.0000,10.0000,10.0000


In [4]:
# Check missing values
print("=== Missing Values ===")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "✅ No missing values found")

# Check duplicates
print("\n=== Duplicate Rows ===")
dupes = df.duplicated().sum()
print(f"✅ No duplicates found" if dupes == 0 else f"⚠️ {dupes} duplicate rows found")

=== Missing Values ===
✅ No missing values found

=== Duplicate Rows ===
✅ No duplicates found


In [5]:
# Check outliers using IQR method on key numeric columns
print("=== Outlier Check (IQR Method) ===")

cols_to_check = ['Price_in_Lakhs', 'Size_in_SqFt', 'Price_per_SqFt']

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f"{col}: {len(outliers):,} outliers ({len(outliers)/len(df)*100:.1f}%) | Range: [{lower:.4f}, {upper:.4f}]")

# Check Floor_No logic issue
print("\n=== Logic Check: Floor_No > Total_Floors ===")
bad_floors = df[df['Floor_No'] > df['Total_Floors']]
print(f"⚠️ {len(bad_floors):,} records where Floor_No > Total_Floors")

=== Outlier Check (IQR Method) ===
Price_in_Lakhs: 0 outliers (0.0%) | Range: [-233.9450, 743.3750]
Size_in_SqFt: 0 outliers (0.0%) | Range: [-1753.5000, 7250.5000]
Price_per_SqFt: 20,020 outliers (8.0%) | Range: [-0.1150, 0.3250]

=== Logic Check: Floor_No > Total_Floors ===
⚠️ 116,304 records where Floor_No > Total_Floors


In [6]:
# Fix 1: Recalculate Price_per_SqFt properly (original values were pre-rounded)
df['Price_per_SqFt'] = (df['Price_in_Lakhs'] / df['Size_in_SqFt']).round(4)
print("✅ Price_per_SqFt recalculated")

# Fix 2: Clip Price_per_SqFt outliers using IQR bounds
Q1 = df['Price_per_SqFt'].quantile(0.25)
Q3 = df['Price_per_SqFt'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
df['Price_per_SqFt'] = df['Price_per_SqFt'].clip(lower=lower, upper=upper)
print(f"✅ Price_per_SqFt outliers clipped to [{lower:.4f}, {upper:.4f}]")

# Fix 3: Cap Floor_No at Total_Floors
df['Floor_No'] = df[['Floor_No', 'Total_Floors']].min(axis=1)
print(f"✅ Floor_No capped at Total_Floors for all records")

# Fix 4: Convert Amenities string into a count (usable numeric feature)
df['Amenity_Count'] = df['Amenities'].apply(
    lambda x: len([a.strip() for a in x.split(',') if a.strip()])
)
print(f"✅ Amenity_Count created from Amenities column")

# Fix 5: Drop ID column (not a feature)
df.drop(columns=['ID'], inplace=True)
print(f"✅ ID column dropped")

print(f"\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

✅ Price_per_SqFt recalculated
✅ Price_per_SqFt outliers clipped to [-0.1198, 0.3277]
✅ Floor_No capped at Total_Floors for all records
✅ Amenity_Count created from Amenities column
✅ ID column dropped

Final shape: 250,000 rows × 23 columns


In [7]:
# Save cleaned dataset
df.to_csv("india_housing_cleaned.csv", index=False)

print("✅ Cleaned file saved as: india_housing_cleaned.csv")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")

✅ Cleaned file saved as: india_housing_cleaned.csv
Shape: 250,000 rows × 23 columns
Missing values: 0
Duplicates: 0


## Phase 1 Summary — Data Cleaning

### Dataset Overview
- **Total Records:** 250,000 rows × 23 columns
- **Missing Values:** None
- **Duplicate Rows:** None

### Issues Found & Fixed

| # | Issue | Action Taken |
|---|-------|-------------|
| 1 | `Price_per_SqFt` was pre-rounded, losing precision | Recalculated as `Price_in_Lakhs / Size_in_SqFt` |
| 2 | 20,020 outliers (8%) in `Price_per_SqFt` | Clipped using IQR bounds |
| 3 | 116,304 records where `Floor_No > Total_Floors` | Capped `Floor_No` at `Total_Floors` |
| 4 | `Amenities` was a raw comma-separated string | Created `Amenity_Count` as a numeric feature |
| 5 | `ID` column not useful for modeling | Dropped |

### Output
- Cleaned file saved as `india_housing_cleaned.csv` — ready for EDA